# Message-kind classifier: labeling, evaluation and ML baseline

The classifier used by the project is **rule-based**: `src.utils.job_filter.classify_post` scores a post with
regex evidence (hiring phrases, pay, working terms, job types) and labels it one of
`job_offer, job_seeker, cargo, travel, sale, service, chat`. Only `job_offer` posts reach the dashboard,
so the number that matters most is precision / recall of `job_offer`.

Nothing in the repo is labeled, so this notebook:

1. draws a sample of real messages and exports it for **hand labeling** (`data/labels/classifier_sample.csv`);
2. lets you label it (in a loop here, or in any text editor — **not Excel**, it rounds long ids);
3. **evaluates the rule-based classifier** against your labels (per-class P/R/F1, confusion matrix, errors);
4. **trains an ML baseline** (TF-IDF + logistic regression) on the same labels and compares it with the rules,
   with and without the rule output as extra features, using the same cross-validation folds.

The ML model is a comparison only; production keeps using the rules until a model clearly beats them.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "utils").is_dir():
        sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the project root")

import warnings

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

from config.settings import settings
from src.preprocess import load_messages
from src.utils.job_filter import JOB_OFFER, KINDS, classify_post, job_score, normalize

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 160)
SEED = 42
LABELS_DIR = PROJECT_ROOT / "data" / "labels"
SAMPLE_CSV = LABELS_DIR / "classifier_sample.csv"
MODEL_PATH = PROJECT_ROOT / "data" / "cache" / "kind_model.pkl"

## 1. Draw the sample

`load_messages(jobs_only=False)` reads every `data/raw/*.jsonl` file, drops reposts and media-only posts and adds the
rule-based `kind`. The sample has two parts:

- **random** — uniform over all messages. Because it keeps the real class balance, the metrics computed on it
  (precision especially) reflect what the dashboard sees.
- **boost** — extra messages from the rarer predicted kinds and from near-threshold scores, so every class has
  enough labels and the borderline cases get checked.

The CSV is only written if it does not exist yet, so re-running never overwrites your labels.

In [ ]:
N_RANDOM = 500   # uniform part
N_BOOST_PER_KIND = 40  # extra per predicted kind (fewer if the kind is rare)
N_BORDERLINE = 80      # extra posts whose score is close to the job-offer threshold

messages = load_messages(jobs_only=False)
messages["rule_score"] = messages["text"].map(job_score)
print(f"{len(messages):,} unique messages")
print(messages["kind"].value_counts().to_string())

In [ ]:
def draw_sample(messages: pd.DataFrame) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    random_part = messages.sample(min(N_RANDOM, len(messages)), random_state=SEED).assign(part="random")
    rest = messages.drop(random_part.index)
    boost = [
        group.sample(min(N_BOOST_PER_KIND, len(group)), random_state=SEED)
        for _, group in rest.groupby("kind")
    ]
    rest = rest.drop(pd.concat(boost).index)
    near = rest.loc[(rest["rule_score"] - 3.0).abs().sort_values().index[:N_BORDERLINE * 3]]
    boost.append(near.sample(min(N_BORDERLINE, len(near)), random_state=SEED))
    sample = pd.concat([random_part, pd.concat(boost).assign(part="boost")])
    sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffled: no class order while labeling
    sample.insert(0, "sample_id", range(len(sample)))
    sample = sample.rename(columns={"kind": "rule_kind"})
    sample["label"] = ""
    return sample[["sample_id", "source_file", "msg_id", "date", "part", "rule_kind", "rule_score", "text", "label"]]


if SAMPLE_CSV.exists():
    print(f"{SAMPLE_CSV} already exists — keeping it ({len(pd.read_csv(SAMPLE_CSV)):,} rows)")
else:
    LABELS_DIR.mkdir(parents=True, exist_ok=True)
    draw_sample(messages).to_csv(SAMPLE_CSV, index=False)
    print(f"wrote {SAMPLE_CSV}")

## 2. Label it

Label guide (what the dashboard needs: a post **offering** a job):

| code | label | meaning |
|---|---|---|
| `o` | `job_offer` | someone is hiring / has work to give (also short "1 kishi kerak", shift notices) |
| `s` | `job_seeker` | someone looks for work ("ish bormi?", "ish kerak") |
| `c` | `cargo` | parcels, luggage, delivery services |
| `t` | `travel` | tickets, flights, car seats, rides |
| `a` | `sale` | things for sale, cars, SIM cards, rooms and flats for rent |
| `v` | `service` | courses, money transfer, loans, visa/paperwork, taxi and other services |
| `h` | `chat` | everything else: greetings, questions, news, rules |

Labels are saved after every answer, so you can stop (`q`) and continue later. `u` undoes the last answer, `p` skips.
The `rule_kind` is hidden while labeling so it does not bias you. You can also edit the `label` column of the CSV
directly (text editor, VS Code or Google Sheets — **not Excel**).

In [ ]:
CODES = {"o": "job_offer", "s": "job_seeker", "c": "cargo", "t": "travel", "a": "sale", "v": "service", "h": "chat"}
assert set(CODES.values()) == set(KINDS)


def label_loop(max_items: int | None = None) -> None:
    from IPython.display import clear_output

    sample = pd.read_csv(SAMPLE_CSV, dtype={"label": str}, keep_default_na=False)
    todo = sample.index[sample["label"].eq("")].tolist()[:max_items]
    done_now: list[int] = []
    for position, index in enumerate(todo):
        clear_output(wait=True)
        labeled = int(sample["label"].ne("").sum())
        print(f"[{labeled}/{len(sample)} labeled]  codes: " + "  ".join(f"{k}={v}" for k, v in CODES.items()) + "  p=skip u=undo q=quit\n")
        print(sample.at[index, "text"][:1500])
        answer = input("\nlabel> ").strip().lower()
        if answer == "q":
            break
        if answer == "u" and done_now:
            sample.at[done_now.pop(), "label"] = ""
            todo.insert(position + 1, index)  # show it again after the undone one
            continue
        if answer in CODES:
            sample.at[index, "label"] = CODES[answer]
            done_now.append(index)
            sample.to_csv(SAMPLE_CSV, index=False)
    clear_output(wait=True)
    print(f"{int(sample['label'].ne('').sum())}/{len(sample)} labeled")


# label_loop()          # uncomment to label here; label_loop(50) does 50 at a time

## 3. Load the labels

In [ ]:
sample = pd.read_csv(SAMPLE_CSV, dtype={"label": str}, keep_default_na=False)
sample["label"] = sample["label"].str.strip().str.lower()
unknown = sorted(set(sample["label"]) - set(KINDS) - {""})
assert not unknown, f"unknown labels in the CSV: {unknown}"

labeled = sample[sample["label"].ne("")].reset_index(drop=True)
print(f"{len(labeled)} of {len(sample)} labeled")
assert len(labeled) >= 50, "label at least ~200 messages (more is better) before evaluating"

# The rules are cheap: re-run them now so the evaluation always uses the current job_filter.py
labeled["pred"] = labeled["text"].map(classify_post)
labeled["score"] = labeled["text"].map(job_score)

counts = labeled.groupby(["label", "part"]).size().unstack(fill_value=0)
counts["total"] = counts.sum(axis=1)
counts.reindex(list(KINDS), fill_value=0)

## 4. Evaluate the rule-based classifier

Two views: **all labeled rows** (every class has support, but the rare ones are over-represented) and the
**random part only** (real class balance, so precision and accuracy mean what they would on live data).

In [ ]:
present = [kind for kind in KINDS if kind in set(labeled["label"]) | set(labeled["pred"])]

print("=== all labeled rows ===")
print(classification_report(labeled["label"], labeled["pred"], labels=present, zero_division=0, digits=3))

random_rows = labeled[labeled["part"].eq("random")]
print(f"=== random part only ({len(random_rows)} rows, real class balance) ===")
print(classification_report(random_rows["label"], random_rows["pred"], labels=present, zero_division=0, digits=3))

In [ ]:
def job_offer_metrics(frame: pd.DataFrame, pred_col: str = "pred") -> dict:
    y, p = frame["label"].eq(JOB_OFFER), frame[pred_col].eq(JOB_OFFER)
    tp, fp, fn = int((y & p).sum()), int((~y & p).sum()), int((y & ~p).sum())
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"n": len(frame), "true_offers": int(y.sum()), "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 3), "recall": round(recall, 3),
            "f1": round(2 * precision * recall / max(precision + recall, 1e-9), 3)}


# job_offer vs everything else — the decision the dashboard depends on
pd.DataFrame({"all labeled": job_offer_metrics(labeled), "random part": job_offer_metrics(random_rows)}).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, normalize_by, title in ((axes[0], None, "counts"), (axes[1], "true", "share of each true class")):
    ConfusionMatrixDisplay.from_predictions(
        labeled["label"], labeled["pred"], labels=present, normalize=normalize_by, ax=ax, cmap="Blues",
        xticks_rotation=40, colorbar=False, values_format=None if normalize_by is None else ".2f",
    )
    ax.set_title(f"Rules vs hand labels — {title}")
    ax.set_xlabel("rule-based prediction")
    ax.set_ylabel("hand label")
plt.tight_layout()
plt.show()

### Errors, worst first

False positives (the rules say job offer, you say no) put noise on the dashboard; false negatives hide real ads. Read these and fix `job_filter.py`.

In [ ]:
def show_errors(frame: pd.DataFrame, label: str, pred: str | None = None, pred_col: str = "pred", n: int = 15) -> pd.DataFrame:
    rows = frame[frame["label"].eq(label) & frame[pred_col].ne(label)]
    if pred is not None:
        rows = rows[rows[pred_col].eq(pred)]
    print(f"{len(rows)} rows labeled {label!r} but not predicted so" + (f" (predicted {pred!r})" if pred else ""))
    return rows[["sample_id", "label", pred_col, "score", "text"]].head(n)


print("-- missed job offers (false negatives)")
display(show_errors(labeled, JOB_OFFER))
print("-- false job offers (false positives)")
fp = labeled[labeled["pred"].eq(JOB_OFFER) & labeled["label"].ne(JOB_OFFER)]
print(f"{len(fp)} rows"); fp[["sample_id", "label", "pred", "score", "text"]].head(15)

In [ ]:
# Error rate by predicted score: is THRESHOLD = 3.0 in the right place?
bins = pd.cut(labeled["score"], [-np.inf, -2, 0, 2, 3, 4, 6, np.inf])
by_score = labeled.groupby(bins, observed=True).agg(n=("label", "size"), share_job_offer=("label", lambda s: (s == JOB_OFFER).mean()))
by_score.round(3)

## 5. ML baseline: TF-IDF + logistic regression

Same labels, same cross-validation folds for every model, so the numbers are comparable:

- **rules** — the existing classifier (nothing to fit);
- **tfidf-lr** — word 1–2-grams and character 2–5-grams of the cleaned Latin-script text (`job_filter.normalize`,
  which transliterates Cyrillic and removes emoji, apostrophes and phone numbers; Hangul is kept) → class-balanced
  logistic regression;
- **tfidf-lr + rules** — the same plus the rule output (predicted kind one-hot and score) as features, which lets the
  model learn when to trust or overrule the rules.

Every score below comes from out-of-fold predictions (`cross_val_predict`), never from rows the model was trained on.

In [ ]:
class RuleFeatures(BaseEstimator, TransformerMixin):
    # The rule output for each raw text: predicted kind (one-hot) and score
    def fit(self, X, y=None):
        self.kinds_ = np.array(KINDS)
        return self

    def transform(self, X):
        texts = pd.Series(X).astype(str)
        kinds = texts.map(classify_post).to_numpy()
        onehot = (kinds[:, None] == self.kinds_[None, :]).astype(float)
        score = texts.map(job_score).to_numpy(dtype=float)[:, None] / 5.0
        return sparse.csr_matrix(np.hstack([onehot, score]))


def text_features() -> FeatureUnion:
    clean = FunctionTransformer(lambda X: [normalize(t) for t in X], validate=False)
    return make_pipeline(clean, FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=60000)),
    ]))


def make_model(with_rules: bool) -> object:
    features = [("text", text_features())]
    if with_rules:
        features.append(("rules", RuleFeatures()))
    return make_pipeline(FeatureUnion(features), LogisticRegression(C=10, max_iter=2000, class_weight="balanced"))


X, y = labeled["text"].to_numpy(dtype=object), labeled["label"].to_numpy()
min_class = pd.Series(y).value_counts().min()
folds = StratifiedKFold(n_splits=int(min(5, max(min_class, 2))), shuffle=True, random_state=SEED)
print(f"{folds.get_n_splits()}-fold CV; rarest class has {min_class} labels")

labeled["pred_rules"] = labeled["pred"]
labeled["pred_ml"] = cross_val_predict(make_model(False), X, y, cv=folds)
labeled["pred_ml_rules"] = cross_val_predict(make_model(True), X, y, cv=folds)

In [ ]:
MODELS = {"rules": "pred_rules", "tfidf-lr": "pred_ml", "tfidf-lr + rules": "pred_ml_rules"}


def compare(frame: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for name, col in MODELS.items():
        jm = job_offer_metrics(frame, col)
        rows[name] = {
            "accuracy": round((frame["label"] == frame[col]).mean(), 3),
            "macro F1": round(f1_score(frame["label"], frame[col], average="macro", zero_division=0), 3),
            "job_offer precision": jm["precision"], "job_offer recall": jm["recall"], "job_offer F1": jm["f1"],
        }
    return pd.DataFrame(rows).T


print("all labeled rows"); display(compare(labeled))
print("random part only (real class balance)"); display(compare(labeled[labeled["part"].eq("random")]))

In [ ]:
# Per-class F1 side by side
per_class = pd.DataFrame({
    name: pd.Series(precision_recall_fscore_support(labeled["label"], labeled[col], labels=present, zero_division=0)[2], index=present)
    for name, col in MODELS.items()
}).round(3)
per_class["support"] = labeled["label"].value_counts().reindex(present)
per_class

In [ ]:
# Where do the models disagree with the labels? Rows the rules get wrong that the ML model gets right, and the reverse.
rules_wrong_ml_right = labeled[(labeled["pred_rules"] != labeled["label"]) & (labeled["pred_ml_rules"] == labeled["label"])]
ml_wrong_rules_right = labeled[(labeled["pred_rules"] == labeled["label"]) & (labeled["pred_ml_rules"] != labeled["label"])]
print(f"rules wrong, tfidf-lr + rules right: {len(rules_wrong_ml_right)}   |   rules right, tfidf-lr + rules wrong: {len(ml_wrong_rules_right)}")
rules_wrong_ml_right[["sample_id", "label", "pred_rules", "pred_ml_rules", "text"]].head(15)

In [ ]:
# The words and character n-grams the plain text model relies on most, per class
plain = make_model(False).fit(X, y)
names = np.array(plain[0].transformer_list[0][1][-1].get_feature_names_out())  # the word + char TF-IDF union
classifier = plain[-1]
for kind, weights in zip(classifier.classes_, classifier.coef_):
    top = np.argsort(weights)[::-1][:12]
    print(f"{kind:11s}", ", ".join(n.split("__", 1)[-1] for n in names[top]))

## 6. Reading the result and saving the model

- If **rules ≥ ML** on the random part, keep the rules; use the error lists above to refine regexes and the
  `THRESHOLD`/weights in `src/utils/job_filter.py`, then re-run sections 3–4 (the rules are re-evaluated on the same labels).
- If **tfidf-lr + rules** is clearly better (a few points of job_offer F1 on the random part, on a few hundred
  labels) it is worth wiring in. More labels help the ML model far more than they help the rules: label another
  batch with `label_loop()` and re-run.
- Keep in mind the size of the sample: with ~50 job offers in the random part, one error moves the metric by about 2 points.

The cell below fits the combined model on all labels and saves it; nothing in the app loads it.

In [ ]:
final = make_model(True).fit(X, y)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
# cloudpickle: the model holds a lambda and a class defined in this notebook, which plain pickle cannot store
with MODEL_PATH.open("wb") as target:
    cloudpickle.dump({"model": final, "classes": list(final.classes_), "n_labels": len(y)}, target)
print(f"saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB), trained on {len(y)} labels")

example = "Zavodga 3 kishiga ish bor. Oylik 2.800.000 won. Uy beriladi"
print(final.predict([example])[0], dict(zip(final.classes_, final.predict_proba([example])[0].round(3))))